# Exp9.0 — Rotating within-user vs cross-user CV

Aggregation-only notebook for protocol `rotating_grouped_cv_v2`. The primary result is pooled out-of-fold (OOF) test performance.

In [1]:
from pathlib import Path
import json
import pandas as pd

def find_repo_root(start=Path.cwd()):
    for path in (start, *start.parents):
        if (path / 'scripts').exists() and (path / 'notebooks').exists():
            return path
    raise RuntimeError('Repository root not found')

repo = find_repo_root()
root = repo / 'notebooks' / 'artifacts' / 'experiment_9_0_within_user_generalization' / 'rotating_grouped_cv_v2'
audit = json.loads((root / 'audit.json').read_text())
audit


{'cross_user_split_unit': 'user',
 'cross_user_user_grouping': True,
 'expected_runs': 20,
 'experiment_id': 'experiment_9_0_within_user_generalization',
 'model_seeds': [11, 23, 37, 53, 71],
 'n_folds': 5,
 'protocol_version': 'rotating_grouped_cv_v2',
 'rotations': [0, 1, 2, 3, 4],
 'source_trials': 39,
 'target_split': {'test': 0.2, 'train': 0.6, 'val': 0.2},
 'total_classes': 12,
 'total_samples': 853,
 'total_users': 20,
 'within_user_note': 'Segments are split within each user because the dataset has only 1-2 source trials per user; grouping source trials would make within-user 5-fold CV impossible.',
 'within_user_source_trial_grouping': False,
 'within_user_split_unit': 'segment'}

## Fold construction

- `within_user`: segment-level stratified folds inside each user.
- `cross_user`: user-level folds; test users are unseen during training.
- Each rotation uses 3 folds train, 1 validation, 1 test.

In [2]:
fold_summary = pd.read_csv(root / 'fold_summary.csv')
rotation_summary = pd.read_csv(root / 'rotation_summary.csv')
display(fold_summary)
display(rotation_summary)


,cv_mode,cv_fold,n_samples,n_users,n_classes,users
0,within_user,0,169,20,12,user_0|user_1|user_10|user_11|user_12|user_13|...
1,within_user,1,174,20,12,user_0|user_1|user_10|user_11|user_12|user_13|...
2,within_user,2,169,20,12,user_0|user_1|user_10|user_11|user_12|user_13|...
3,within_user,3,171,20,12,user_0|user_1|user_10|user_11|user_12|user_13|...
4,within_user,4,170,20,12,user_0|user_1|user_10|user_11|user_12|user_13|...
5,cross_user,0,174,4,12,user_1|user_15|user_18|user_20
6,cross_user,1,169,4,12,user_10|user_12|user_2|user_6
7,cross_user,2,172,4,12,user_13|user_5|user_7|user_8
8,cross_user,3,168,4,12,user_0|user_14|user_16|user_3
9,cross_user,4,170,4,12,user_11|user_19|user_4|user_9


,cv_mode,rotation,test_fold,val_fold,train_samples,val_samples,test_samples,train_users,val_users,test_users,test_same_user_seen_fraction,test_same_user_class_seen_fraction
0,within_user,0,0,1,510,174,169.0,20,20,20,1.0,0.958580
1,within_user,1,1,2,510,169,174.0,20,20,20,1.0,0.971264
2,within_user,2,2,3,513,171,169.0,20,20,20,1.0,0.970414
3,within_user,3,3,4,512,170,171.0,20,20,20,1.0,0.964912
4,within_user,4,4,0,514,169,170.0,20,20,20,1.0,0.970588
5,cross_user,0,0,1,510,169,174.0,12,4,4,0.0,0.000000
6,cross_user,1,1,2,512,172,169.0,12,4,4,0.0,0.000000
7,cross_user,2,2,3,513,168,172.0,12,4,4,0.0,0.000000
8,cross_user,3,3,4,515,170,168.0,12,4,4,0.0,0.000000
9,cross_user,4,4,0,509,174,170.0,12,4,4,0.0,0.000000


## Primary OOF results

Every sample is held out as test exactly once per CV mode and method. These pooled metrics are the primary comparison.

In [3]:
oof = pd.read_csv(root / 'oof_test_metrics.csv')
gap = pd.read_csv(root / 'within_vs_cross_user.csv')
display(oof.sort_values(['method', 'cv_mode']))
display(gap)


,cv_mode,method,n,accuracy,balanced_accuracy,macro_f1
3,cross_user,a2_234x234,853,0.485346,0.476916,0.471886
1,within_user,a2_234x234,853,0.504103,0.492262,0.484296
2,cross_user,raw250_linear,853,0.592028,0.584319,0.585331
0,within_user,raw250_linear,853,0.643611,0.639477,0.639845


,method,within_user_oof_accuracy,within_user_oof_ba,within_user_oof_macro_f1,cross_user_oof_accuracy,cross_user_oof_ba,cross_user_oof_macro_f1,user_gap_accuracy,user_gap_ba,user_gap_macro_f1
0,raw250_linear,0.643611,0.639477,0.639845,0.592028,0.584319,0.585331,0.051583,0.055158,0.054513
1,a2_234x234,0.504103,0.492262,0.484296,0.485346,0.476916,0.471886,0.018757,0.015346,0.012411


## Fold-level stability

Use fold mean/std as a variance diagnostic, not as the primary pooled estimate.

In [4]:
metric_summary = pd.read_csv(root / 'metric_summary.csv')
metric_runs = pd.read_csv(root / 'metric_runs.csv')
display(metric_summary)
display(metric_runs[metric_runs['split'].eq('test')].sort_values(['cv_mode','method','rotation']))


,cv_mode,method,split,accuracy_mean,accuracy_std,balanced_accuracy_mean,balanced_accuracy_std,macro_f1_mean,macro_f1_std,objective_loss_mean,objective_loss_std,n_mean,n_std
0,within_user,raw250_linear,train,0.998827,0.001070,0.998769,0.001125,0.998819,0.001078,NaN,NaN,511.8,1.788854
1,within_user,raw250_linear,val,0.646631,0.072876,0.643608,0.070206,0.637639,0.072765,NaN,NaN,170.6,2.073644
2,within_user,raw250_linear,test,0.643225,0.058423,0.634908,0.060360,0.630147,0.060643,NaN,NaN,170.6,2.073644
3,within_user,a2_234x234,train,0.766745,0.205479,0.763630,0.209893,0.757202,0.221910,0.966522,0.527350,511.8,1.788854
4,within_user,a2_234x234,val,0.512089,0.090900,0.509139,0.078923,0.493439,0.095209,1.530014,0.270618,170.6,2.073644
5,within_user,a2_234x234,test,0.503651,0.107234,0.495964,0.100969,0.478076,0.116551,1.514309,0.297069,170.6,2.073644
6,cross_user,raw250_linear,train,0.980747,0.043052,0.977810,0.049617,0.979577,0.045668,NaN,NaN,511.8,2.387467
7,cross_user,raw250_linear,val,0.623684,0.052255,0.617120,0.048876,0.613650,0.043521,NaN,NaN,170.6,2.408319
8,cross_user,raw250_linear,test,0.591997,0.050004,0.585251,0.047843,0.582204,0.050817,NaN,NaN,170.6,2.408319
9,cross_user,a2_234x234,train,0.787528,0.181663,0.783954,0.185543,0.781462,0.190865,0.907983,0.460633,511.8,2.387467


,cv_mode,method,rotation,seed,split,accuracy,balanced_accuracy,macro_f1,objective_loss,n,best_epoch,parameter_count,test_same_user_seen_fraction,test_same_user_class_seen_fraction
47,cross_user,a2_234x234,0,11,test,0.517241,0.508664,0.501077,1.511255,174,94.0,21760,0.0,0.000000
50,cross_user,a2_234x234,1,23,test,0.526627,0.509941,0.498533,1.419427,169,99.0,21760,0.0,0.000000
53,cross_user,a2_234x234,2,37,test,0.348837,0.345534,0.322536,1.997684,172,97.0,21760,0.0,0.000000
56,cross_user,a2_234x234,3,53,test,0.571429,0.548015,0.536193,1.485090,168,99.0,21760,0.0,0.000000
59,cross_user,a2_234x234,4,71,test,0.464706,0.460165,0.456902,1.609084,170,88.0,21760,0.0,0.000000
32,cross_user,raw250_linear,0,11,test,0.637931,0.630933,0.631192,NaN,174,NaN,5760,0.0,0.000000
35,cross_user,raw250_linear,1,23,test,0.556213,0.551447,0.544201,NaN,169,NaN,5760,0.0,0.000000
38,cross_user,raw250_linear,2,37,test,0.558140,0.554586,0.553359,NaN,172,NaN,5760,0.0,0.000000
41,cross_user,raw250_linear,3,53,test,0.654762,0.643683,0.643546,NaN,168,NaN,5760,0.0,0.000000
44,cross_user,raw250_linear,4,71,test,0.552941,0.545604,0.538724,NaN,170,NaN,5760,0.0,0.000000


## Per-user and per-class OOF diagnostics

In [5]:
per_user = pd.read_csv(root / 'oof_per_user_test.csv')
per_class = pd.read_csv(root / 'oof_per_class_test.csv')
display(per_user.sort_values(['cv_mode','method','balanced_accuracy']))
display(per_class.sort_values(['cv_mode','method','class_label']))


,cv_mode,method,user,n,accuracy,balanced_accuracy,macro_f1
75,cross_user,a2_234x234,user_5,42,0.238095,0.229167,0.222421
77,cross_user,a2_234x234,user_7,55,0.290909,0.288889,0.237831
60,cross_user,a2_234x234,user_0,22,0.318182,0.291667,0.243056
69,cross_user,a2_234x234,user_18,30,0.266667,0.305556,0.229365
79,cross_user,a2_234x234,user_9,40,0.300000,0.305556,0.291865
...,...,...,...,...,...,...,...
11,within_user,raw250_linear,user_2,44,0.750000,0.750000,0.741270
6,within_user,raw250_linear,user_14,41,0.780488,0.756944,0.751190
1,within_user,raw250_linear,user_1,60,0.850000,0.811111,0.793696
14,within_user,raw250_linear,user_4,44,0.840909,0.833333,0.824272


,cv_mode,method,class_label,class_index,precision,recall,f1,support
36,cross_user,a2_234x234,A,0,0.421053,0.363636,0.390244,66
37,cross_user,a2_234x234,B,1,0.659091,0.408451,0.504348,71
38,cross_user,a2_234x234,C,2,0.616279,0.679487,0.646341,78
39,cross_user,a2_234x234,D,3,0.390000,0.527027,0.448276,74
40,cross_user,a2_234x234,E,4,0.568182,0.657895,0.609756,76
41,cross_user,a2_234x234,G,5,0.441176,0.625000,0.517241,72
42,cross_user,a2_234x234,H,6,0.480000,0.600000,0.533333,80
43,cross_user,a2_234x234,I,7,0.411765,0.276316,0.330709,76
44,cross_user,a2_234x234,J,8,0.476190,0.266667,0.341880,75
45,cross_user,a2_234x234,K,9,0.638889,0.582278,0.609272,79


## Pooled OOF confusion matrices

In [6]:
conf = pd.read_csv(root / 'confusion_summary.csv')
for (cv_mode, method), frame in conf.groupby(['cv_mode','method'], sort=False):
    print(f'\n{cv_mode} / {method}')
    display(frame.pivot(index='true_label', columns='pred_label', values='count'))



within_user / raw250_linear


pred_label,A,B,C,D,E,G,H,I,J,K,L,X
true_label,,,,,,,,,,,,
A,31,2,0,2,4,4,16,2,1,3,0,1
B,2,44,0,6,6,5,3,1,0,4,0,0
C,0,0,57,0,0,4,0,5,6,0,6,0
D,1,5,1,46,2,3,4,2,1,4,1,4
E,4,6,0,0,54,7,2,0,1,0,0,2
G,0,1,6,1,3,47,4,2,3,4,1,0
H,11,2,0,2,3,1,54,0,0,4,0,3
I,2,0,0,0,4,0,2,50,14,2,1,1
J,0,0,2,0,0,0,1,17,50,0,2,3



within_user / a2_234x234


pred_label,A,B,C,D,E,G,H,I,J,K,L,X
true_label,,,,,,,,,,,,
A,25,6,0,5,2,7,9,2,0,7,1,2
B,4,50,0,2,2,3,7,1,0,0,1,1
C,1,0,58,0,1,5,1,2,2,1,6,1
D,2,14,1,31,0,1,7,4,6,1,3,4
E,2,7,1,0,53,7,2,2,1,1,0,0
G,8,3,3,1,9,40,0,1,2,4,1,0
H,6,2,0,4,1,3,51,3,1,4,2,3
I,1,2,2,5,5,2,10,18,18,0,10,3
J,0,1,5,8,3,3,7,14,26,1,5,2



cross_user / raw250_linear


pred_label,A,B,C,D,E,G,H,I,J,K,L,X
true_label,,,,,,,,,,,,
A,33,1,0,0,5,4,15,3,1,4,0,0
B,6,38,1,12,1,6,2,1,1,3,0,0
C,0,0,58,0,0,1,0,2,9,0,8,0
D,1,4,1,45,2,1,5,2,4,3,0,6
E,4,8,0,0,52,1,4,2,1,3,0,1
G,1,1,10,1,2,43,3,3,2,5,1,0
H,9,1,0,7,3,2,50,1,0,3,1,3
I,1,2,1,0,7,1,0,42,15,3,3,1
J,1,0,2,5,2,0,0,13,46,0,3,3



cross_user / a2_234x234


pred_label,A,B,C,D,E,G,H,I,J,K,L,X
true_label,,,,,,,,,,,,
A,24,1,1,8,5,5,7,1,0,6,0,8
B,3,29,0,13,4,9,5,2,0,5,0,1
C,2,0,53,2,0,8,0,2,1,0,8,2
D,4,3,3,39,1,2,4,1,3,1,2,11
E,3,4,1,1,50,11,2,0,0,3,0,1
G,2,0,7,1,7,45,5,3,0,2,0,0
H,4,3,1,6,5,4,48,0,0,1,0,8
I,2,0,5,8,8,2,7,21,15,0,5,3
J,0,1,7,8,5,4,8,13,20,1,6,2
